# Five-minute plcopen digital twin

This notebook runs one simulated axis with the same motion kernel used by the C++ library. It uses no hardware, wall-clock loop, or plotting dependency.

In [ ]:
%pip install pyplcopen==0.20.0

In [ ]:
import pyplcopen

cycle = pyplcopen.CycleConfig.at_1khz()
velocity = cycle.velocity_to_cycle(100.0)       # 100 units/s
acceleration = cycle.acceleration_to_cycle(500.0)
jerk = cycle.jerk_to_cycle(5000.0)

In [ ]:
axis = pyplcopen.AxisSim()
axis.power_on()
axis.home_direct(0.0)
axis.move_velocity(velocity, acceleration, acceleration, jerk, cycles=20)

samples = []
for _ in range(20):
    axis.cycle(25)
    samples.append(axis.command_position())

axis.halt(acceleration, jerk)
assert axis.status() == pyplcopen.AxisStatus.STANDSTILL
assert samples[-1] > samples[0] >= 0.0

In [ ]:
levels = "▁▂▃▄▅▆▇█"
low, high = min(samples), max(samples)
span = high - low or 1.0
sparkline = "".join(levels[min(7, int((value - low) * 8 / span))] for value in samples)
print("command position:", sparkline)
print(f"final position={axis.command_position():.6f}, samples={len(samples)}")